# Hair 6클래스: 원본·증강 × B0·256·B1·384 — 4개 실험

이 노트북은 기존 Hair 5클래스 ZIP과 새 `양호` ZIP을 **읽기 전용**으로 사용해
별도 Hair 6클래스 데이터 ZIP을 만듭니다. 기존 사진은 정제된 Train/Validation/Test
분할을 유지합니다. 새 `양호` 사진은 파일명 첫 숫자 ID로 사람을 묶어 약 80:10:10으로
나눕니다. 기존 Hair 파일명은 사람 ID가 없어 사람·세션 간 누수는 확인되지 않았습니다.

Train 증강은 기존 질환 클래스에서 각 클래스 원본의 약 50%를 추가 생성하고,
적은 `양호` Train에는 원본당 5개의 약한 변형을 추가합니다. 밝기·대비·색감·작은 회전·
좌우 반전·아주 약한 블러만 사용하며 Validation/Test에는 증강을 적용하지 않습니다.
증강 수량은 원본 사람 수가 증가했다는 의미가 아닙니다.

| 실험 | 모델 | 입력 | Loss | Train |
|---|---|---:|---|---|
| 1 | EfficientNet-B0 | 256 | CE | 원본 |
| 2 | EfficientNet-B0 | 256 | CE | 원본 + 증강 |
| 3 | EfficientNet-B1 | 384 | Label Smoothing 0.05 | 원본 |
| 4 | EfficientNet-B1 | 384 | Label Smoothing 0.05 | 원본 + 증강 |

각 실험은 ImageNet 가중치에서 새 6클래스 출력층으로 시작하며, head-only 15 epoch와
후반부 30개 층 미세조정 15 epoch를 실시합니다. **같은 모델의 원본·증강 두 실험은
Train 이미지와 그 수량에 따른 클래스 가중치만 다릅니다.** B0와 B1은
구조·해상도·Loss가 함께 달라 실용적인
모델 구성 비교이지 단일 변수의 인과 비교가 아닙니다. 네 실험을 같은 Validation의
Macro F1로 비교하고, 승자 하나만 고정 Test에서 평가합니다.

입력 ZIP이나 저장 모델은 덮어쓰지 않습니다. 데이터 ZIP은 Drive의 `datasets`, 전체
모델·로그·그래프는 `2_results/hair`에 저장합니다. 모델 없는 작은 보고서 ZIP도
같은 결과 폴더 옆에 만듭니다. 런타임이 끊기면 아래 `RESUME_DIR`에 출력된 결과 폴더를
넣고 처음부터 실행하면 완료 실험을 검증해 건너뜁니다.


<!-- mediflow-notebook-guide-v1 -->
### 실행 안내

- **이 노트북의 목적:** Hair 6클래스 데이터 구성과 네 학습 조건을 실행합니다.
- **실행 순서:** Colab에서 첫 셀부터 아래로 차례대로 실행합니다. 중간 셀만 단독 실행하면 앞에서 만든 설정과 코드가 없어 오류가 날 수 있습니다.
- **수정할 곳:** 앞부분의 경로·실험 설정 셀을 먼저 확인합니다. 내장 실행 코드 셀은 펼쳐서 읽을 수 있지만 보통 수정하지 않습니다.
- **결과 확인:** 끝부분의 저장 경로와 보고서 출력에서 결과를 확인합니다. ZIP과 학습 결과를 혼동하지 않도록 각 단계의 설명을 읽어 주세요.


<!-- mediflow-notebook-sections-v1 -->
## 1. Drive 경로와 재개 설정


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/mediflow_Project')
RESUME_DIR = ''  # 중단 뒤에는 출력된 /content/drive/.../2_results/hair/... 폴더를 입력
SEED = 42
BATCH_SIZE = 32
STAGE1_EPOCHS = 15
STAGE2_EPOCHS = 15

# 원본 데이터 SHA-256: 코드에 검증된 값으로 고정되어 있습니다.
# 새 ZIP은 양호 사진의 6개 증상값이 모두 0이라는 사용자 확인을 라벨 근거로 기록합니다.


<!-- mediflow-notebook-sections-v1 -->
## 2. Drive 연결과 실행 환경


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install tensorflow==2.20.0 keras==3.13.2 pillow matplotlib

import hashlib
import json
import platform
import shutil
import uuid
import zipfile
from datetime import datetime, timezone

import keras
import tensorflow as tf

if tf.__version__ != '2.20.0' or keras.__version__ != '3.13.2':
    raise RuntimeError('설치 버전이 다릅니다. 런타임을 다시 시작하고 처음부터 실행하세요.')
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Colab 런타임에서 GPU를 선택하세요.')
dataset_dir = PROJECT_ROOT / 'datasets'
candidates = [dataset_dir / 'hair_datasets.zip', dataset_dir / 'hair_datasets']
valid = [p for p in candidates if p.is_file() and zipfile.is_zipfile(p)]
if len(valid) != 1:
    available = sorted(p.name for p in dataset_dir.iterdir()) if dataset_dir.is_dir() else []
    raise FileNotFoundError(f'기존 Hair ZIP 한 개를 확인하세요: {available}')
OLD_ZIP = valid[0]
GOOD_ZIP = dataset_dir / 'hair_good_raw_v1.zip'
if not GOOD_ZIP.is_file() or not zipfile.is_zipfile(GOOD_ZIP):
    raise FileNotFoundError(GOOD_ZIP)
print('기존 ZIP:', OLD_ZIP)
print('양호 ZIP:', GOOD_ZIP)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 3. 재현 코드

필요한 코드를 노트북 안에 포함했습니다.


<!-- mediflow-notebook-sections-v1 -->
## 1. Drive 경로와 재개 설정


In [ ]:
from pathlib import Path
_EMBEDDED_SOURCE_DIR = Path('/content/mediflow_notebook_sources/17_hair_six_class_four_experiments_colab')
_EMBEDDED_SOURCE_DIR.mkdir(parents=True, exist_ok=True)


<!-- mediflow-embedded-module-v1 -->
### 내장 실행 코드: `common_engine.py`

이 셀의 코드는 줄 단위로 읽을 수 있도록 펼쳐 놓았습니다. Colab 임시 공간에 기록한 뒤 아래 실행 셀에서 원본과 동일한 SHA-256을 확인합니다. 설정 변경은 앞쪽 설정 셀에서 하세요.


In [ ]:
%%writefile /content/mediflow_notebook_sources/17_hair_six_class_four_experiments_colab/common_engine.py
"""Sequential domain-configured experiments; validation selection precedes any test inference.

The Colab notebook embeds an exact copy of this module so no repository checkout
is needed in Colab. Completed trials are reused only after artifact verification.
Interrupted attempts are preserved and restarted, not resumed mid-epoch.
"""

from __future__ import annotations

import csv
import hashlib
import json
import time
import uuid
from pathlib import Path

import keras
import numpy as np

PROTOCOL = "mediflow_common_v1"
TRIALS = [
    {"id": "b0_224_ce", "backbone": "B0", "size": 224, "loss": "ce"},
    {"id": "b0_256_ce", "backbone": "B0", "size": 256, "loss": "ce"},
    {"id": "b0_256_ls005", "backbone": "B0", "size": 256, "loss": "ls005"},
    {"id": "b0_256_focal15", "backbone": "B0", "size": 256, "loss": "focal15"},
    {"id": "b1_256_ls005", "backbone": "B1", "size": 256, "loss": "ls005"},
]


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(".json-" + uuid.uuid4().hex[:12] + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def classification_metrics(truth, probabilities, count=5):
    truth = np.asarray(truth, dtype=np.int64)
    probabilities = np.asarray(probabilities)
    if (
        truth.ndim != 1
        or not len(truth)
        or probabilities.shape != (len(truth), count)
        or not np.isfinite(probabilities).all()
        or np.any(truth < 0)
        or np.any(truth >= count)
    ):
        raise ValueError("Invalid evaluation arrays")
    predictions = probabilities.argmax(axis=1)
    cm = np.bincount(count * truth + predictions, minlength=count * count).reshape(count, count)
    tp = np.diag(cm).astype(float)
    precision = np.divide(tp, cm.sum(0), out=np.zeros(count), where=cm.sum(0) != 0)
    recall = np.divide(tp, cm.sum(1), out=np.zeros(count), where=cm.sum(1) != 0)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(count),
        where=precision + recall != 0,
    )
    return {
        "accuracy": float(np.mean(truth == predictions)),
        "macro_f1": float(f1.mean()),
        "class_f1": f1.tolist(),
        "precision": precision.tolist(),
        "recall": recall.tolist(),
        "support": cm.sum(1).tolist(),
        "confusion_matrix": cm.tolist(),
        "count": len(truth),
    }


def predict_dataset(model, dataset):
    truth, probabilities = [], []
    for images, labels in dataset:
        probabilities.extend(model(images, training=False).numpy())
        truth.extend(np.argmax(labels.numpy(), axis=1))
    return np.asarray(truth, dtype=np.int64), np.asarray(probabilities)


def save_predictions(path, paths, truth, probabilities):
    if len(paths) != len(truth):
        raise ValueError("File order and prediction count differ")
    with Path(path).open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.writer(stream)
        writer.writerow(
            [
                "path",
                "true_index",
                "pred_index",
                *[f"prob_C{i}" for i in range(probabilities.shape[1])],
            ]
        )
        for name, target, probs in zip(paths, truth, probabilities, strict=True):
            writer.writerow([name, int(target), int(probs.argmax()), *map(float, probs)])


def loss_function(name):
    if name == "ce":
        return keras.losses.CategoricalCrossentropy()
    if name == "ls005":
        return keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
    if name == "focal15":
        return keras.losses.CategoricalFocalCrossentropy(alpha=1.0, gamma=1.5)
    raise ValueError(name)


def build_model(spec):
    builder = {
        "B0": keras.applications.EfficientNetB0,
        "B1": keras.applications.EfficientNetB1,
        "V2S": keras.applications.EfficientNetV2S,
    }
    size = spec["size"]
    backbone = builder[spec["backbone"]](
        include_top=False, weights="imagenet", input_shape=(size, size, 3)
    )
    backbone.trainable = False
    inputs = keras.Input((size, size, 3))
    features = backbone(inputs, training=False)
    features = keras.layers.GlobalAveragePooling2D()(features)
    features = keras.layers.Dropout(0.3)(features)
    outputs = keras.layers.Dense(spec["class_count"], activation="softmax")(features)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss=loss_function(spec["loss"]),
        metrics=["accuracy"],
    )
    return model


def configure_partial(model, loss_name):
    backbones = [
        layer
        for layer in model.layers
        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower()
    ]
    if len(backbones) != 1:
        raise ValueError("Expected one EfficientNet backbone")
    backbone = backbones[0]
    backbone.trainable = True
    for index, layer in enumerate(backbone.layers):
        layer.trainable = index >= len(backbone.layers) - 30 and not isinstance(
            layer, keras.layers.BatchNormalization
        )
    model.compile(
        optimizer=keras.optimizers.Adam(1e-5), loss=loss_function(loss_name), metrics=["accuracy"]
    )
    return [layer.name for layer in backbone.layers if layer.trainable]


class HistoryBackup(keras.callbacks.Callback):
    def __init__(self, path):
        super().__init__()
        self.path = path
        self.values = {}

    def on_epoch_end(self, epoch, logs=None):
        for key, value in (logs or {}).items():
            self.values.setdefault(key, []).append(float(value))
        write_json(self.path, self.values)


def fit_stage(model, train, val, directory, name, epochs):
    best = directory / (name + "_best.keras")
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            str(best), monitor="val_accuracy", mode="max", save_best_only=True
        ),
        keras.callbacks.CSVLogger(str(directory / (name + "_log.csv"))),
        HistoryBackup(directory / (name + "_history.json")),
        keras.callbacks.TerminateOnNaN(),
    ]
    history = model.fit(train, validation_data=val, epochs=epochs, callbacks=callbacks, verbose=2)
    values = {key: [float(v) for v in seq] for key, seq in history.history.items()}
    if len(values.get("val_accuracy", [])) != epochs or not all(
        np.isfinite(seq).all() for seq in values.values()
    ):
        raise RuntimeError("Incomplete or non-finite training; attempt retained")
    model.save(directory / (name + "_last.keras"))
    return values, best


def checkpoint_choice(baseline_score, new_score):
    """Keep the earlier/simpler checkpoint on ties."""
    return new_score > baseline_score


def cached_record(root, trial_id, signature):
    trial_dir = Path(root) / trial_id
    marker = trial_dir / "completed.json"
    if not marker.exists():
        return None
    record = read_json(marker)
    if record["signature"] != signature:
        raise ValueError("Resume settings differ; use a new suite directory")
    for relative, digest in record["artifact_hashes"].items():
        target = (trial_dir / relative).resolve()
        if not target.is_relative_to(trial_dir.resolve()) or file_hash(target) != digest:
            raise ValueError("Completed artifact changed or corrupted: " + relative)
    return record


def evaluate_to_files(model, dataset, paths, directory, prefix):
    truth, probabilities = predict_dataset(model, dataset)
    metrics = classification_metrics(truth, probabilities, probabilities.shape[1])
    write_json(directory / (prefix + "_metrics.json"), metrics)
    save_predictions(directory / (prefix + "_predictions.csv"), paths, truth, probabilities)
    return metrics


def run_trial(spec, dataset_factory, root, signature, seed=42, epochs1=15, epochs2=10):
    cached = cached_record(root, spec["id"], signature)
    if cached:
        return cached
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    directory = Path(root) / spec["id"] / ("attempt_" + uuid.uuid4().hex[:12])
    directory.mkdir(parents=True, exist_ok=False)
    write_json(directory / "spec.json", spec)
    train, _ = dataset_factory("train", spec["size"], True)
    val, paths = dataset_factory("val", spec["size"], False)
    started = time.monotonic()
    model = build_model(spec)
    h1, best1 = fit_stage(model, train, val, directory, "stage1", epochs1)
    del model
    keras.backend.clear_session()
    model = keras.models.load_model(best1, compile=False)
    trainable = []
    h2 = {key: [] for key in h1}
    best2 = best1
    if epochs2:
        trainable = configure_partial(model, spec["loss"])
        h2, best2 = fit_stage(model, train, val, directory, "stage2", epochs2)
    del model
    selected_stage = (
        "stage2"
        if checkpoint_choice(max(h1["val_accuracy"]), max(h2["val_accuracy"], default=-1.0))
        else "stage1"
    )
    selected = best2 if selected_stage == "stage2" else best1
    model = keras.models.load_model(selected, compile=False)
    metrics = evaluate_to_files(model, val, paths, directory, "validation")
    record = {
        "id": spec["id"],
        "spec": spec,
        "signature": signature,
        "attempt": directory.name,
        "selected_model": selected.name,
        "selected_stage": selected_stage,
        "validation": metrics,
        "stage1_best_val": max(h1["val_accuracy"]),
        "stage2_best_val": max(h2["val_accuracy"]) if h2["val_accuracy"] else None,
        "training_seconds": time.monotonic() - started,
        "parameters": model.count_params(),
        "model_bytes": selected.stat().st_size,
        "trainable_backbone_layers": trainable,
        "history": {key: h1[key] + h2[key] for key in h1},
        "stage_boundary": len(h1["accuracy"]),
    }
    finish_record(directory, record)
    return record


def finish_record(directory, record):
    write_json(directory / "record.json", record)
    record["artifact_hashes"] = {
        str(path.relative_to(directory.parent)): file_hash(path)
        for path in directory.iterdir()
        if path.is_file()
    }
    write_json(directory.parent / "completed.json", record)


def selected_model_path(root, record):
    return Path(root) / record["id"] / record["attempt"] / record["selected_model"]


def extend_b1(parent, dataset_factory, root, signature, seed=42, epochs=5):
    trial_id = "b1_256_ls005_extend5"
    cached = cached_record(root, trial_id, signature)
    if cached:
        return cached
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    directory = Path(root) / trial_id / ("attempt_" + uuid.uuid4().hex[:12])
    directory.mkdir(parents=True, exist_ok=False)
    parent_dir = Path(root) / parent["id"] / parent["attempt"]
    # Continue from epoch 10's LAST checkpoint, including optimizer state.
    source = parent_dir / "stage2_last.keras"
    model = keras.models.load_model(source)
    if model.optimizer is None:
        raise ValueError("Extension requires saved optimizer")
    train, _ = dataset_factory("train", 256, True)
    val, paths = dataset_factory("val", 256, False)
    started = time.monotonic()
    history, best = fit_stage(model, train, val, directory, "extension", epochs)
    del model
    parent_best = selected_model_path(root, parent)
    parent_score = max(parent["stage1_best_val"], parent["stage2_best_val"])
    keep_extension = checkpoint_choice(parent_score, max(history["val_accuracy"]))
    selected = directory / "selected.keras"
    import shutil

    shutil.copyfile(best if keep_extension else parent_best, selected)
    model = keras.models.load_model(selected, compile=False)
    metrics = evaluate_to_files(model, val, paths, directory, "validation")
    record = {
        "id": trial_id,
        "spec": {**parent["spec"], "id": trial_id},
        "signature": signature,
        "attempt": directory.name,
        "selected_model": selected.name,
        "selected_stage": "extension" if keep_extension else "parent_" + parent["selected_stage"],
        "validation": metrics,
        "parameters": model.count_params(),
        "model_bytes": selected.stat().st_size,
        "training_seconds": time.monotonic() - started,
        "history": {key: parent["history"][key] + history[key] for key in history},
        "stage_boundary": parent["stage_boundary"],
        "extension_boundary": len(parent["history"]["accuracy"]),
        "parent_last_sha256": file_hash(source),
        "optimizer_restored": True,
        "extension_best_val": max(history["val_accuracy"]),
        "stage1_best_val": parent["stage1_best_val"],
        "stage2_best_val": parent["stage2_best_val"],
    }
    finish_record(directory, record)
    return record


def select_winner(records):
    # Stable order preserves earlier experiments on exact ties.
    return max(records, key=lambda record: record["validation"]["accuracy"])


<!-- mediflow-embedded-module-v1 -->
### 내장 실행 코드: `hair_six_class_suite.py`

이 셀의 코드는 줄 단위로 읽을 수 있도록 펼쳐 놓았습니다. Colab 임시 공간에 기록한 뒤 아래 실행 셀에서 원본과 동일한 SHA-256을 확인합니다. 설정 변경은 앞쪽 설정 셀에서 하세요.


In [ ]:
%%writefile /content/mediflow_notebook_sources/17_hair_six_class_four_experiments_colab/hair_six_class_suite.py
"""Build an isolated Hair six-class dataset and run four Colab comparisons."""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import shutil
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

from PIL import Image, ImageEnhance, ImageFilter, ImageOps

CLASSES = ["모낭사이홍반", "미세각질", "비듬", "탈모", "피지과다", "양호"]
SPLITS = ("train", "val", "test")
OLD_SHA256 = "2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156"
GOOD_SHA256 = "d602fdaf3ff3b4dbe288603dd1684316f468702ace751dde1c5e1a3e3215b86f"
TRIALS = (
    {"id": "b0_256_original", "backbone": "B0", "size": 256, "loss": "ce", "variant": "original"},
    {"id": "b0_256_augmented", "backbone": "B0", "size": 256, "loss": "ce", "variant": "augmented"},
    {
        "id": "b1_384_original", "backbone": "B1", "size": 384,
        "loss": "ls005", "variant": "original",
    },
    {
        "id": "b1_384_augmented", "backbone": "B1", "size": 384,
        "loss": "ls005", "variant": "augmented",
    },
)
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _safe_members(archive: zipfile.ZipFile):
    seen = set()
    for info in archive.infolist():
        name = info.filename
        path = PurePosixPath(name)
        if path.is_absolute() or ".." in path.parts or "\\" in name or ":" in name:
            raise ValueError(f"Unsafe ZIP path: {name}")
        key = name.casefold()
        if key in seen:
            raise ValueError(f"Repeated ZIP path: {name}")
        seen.add(key)
        if not info.is_dir() and path.suffix.lower() in IMAGE_SUFFIXES:
            yield info


def _class_name(value: str) -> str:
    if value in CLASSES:
        return value
    try:
        repaired = value.encode("cp437").decode("cp949")
    except (UnicodeError, ValueError):
        repaired = value
    if repaired not in CLASSES[:-1]:
        raise ValueError(f"Unexpected Hair class: {value!r}")
    return repaired


def _old_entry(name: str) -> tuple[str, str] | None:
    parts = PurePosixPath(name).parts
    for index, part in enumerate(parts):
        if part.lower() == "original" and len(parts) == index + 4:
            split = parts[index + 1].lower()
            if split in SPLITS:
                return split, _class_name(parts[index + 2])
    return None


def split_good_subjects(groups: dict[str, list[str]], seed: int = 42) -> dict[str, str]:
    """Keep each user-confirmed filename ID wholly in one split."""
    if len(groups) < 3:
        raise ValueError("At least three good subject IDs are needed")
    rng = random.Random(seed)
    subjects = list(groups)
    rng.shuffle(subjects)
    subjects.sort(key=lambda subject: -len(groups[subject]))
    total = sum(map(len, groups.values()))
    targets = {"train": total * 0.8, "val": total * 0.1, "test": total * 0.1}
    counts = dict.fromkeys(SPLITS, 0)
    allocation = {}
    for subject in subjects:
        size = len(groups[subject])

        def cost(split, size=size):
            proposed = {key: counts[key] + (size if key == split else 0) for key in SPLITS}
            return sum((proposed[key] - targets[key]) ** 2 / targets[key] for key in SPLITS)

        chosen = min(SPLITS, key=cost)
        allocation[subject] = chosen
        counts[chosen] += size
    if any(counts[split] == 0 for split in SPLITS):
        raise ValueError(f"Good split is empty: {counts}")
    return allocation


def _pixel_hash(data: bytes) -> str:
    with Image.open(io.BytesIO(data)) as image:
        rgb = image.convert("RGB")
        digest = hashlib.sha256(f"{rgb.width}x{rgb.height}:".encode("ascii"))
        digest.update(rgb.tobytes())
    return digest.hexdigest()


def _augment(source: Path, target: Path, seed: int) -> dict:
    rng = random.Random(seed)
    angle = rng.uniform(-10.0, 10.0)
    brightness = rng.uniform(0.85, 1.15)
    contrast = rng.uniform(0.85, 1.15)
    saturation = rng.uniform(0.90, 1.10)
    mirrored = rng.random() < 0.5
    blur = rng.random() < 0.12
    with Image.open(source) as image:
        result = image.convert("RGB")
        if mirrored:
            result = ImageOps.mirror(result)
        result = result.rotate(
            angle, resample=Image.Resampling.BICUBIC,
            fillcolor=result.getpixel((0, 0)),
        )
        result = ImageEnhance.Brightness(result).enhance(brightness)
        result = ImageEnhance.Contrast(result).enhance(contrast)
        result = ImageEnhance.Color(result).enhance(saturation)
        if blur:
            result = result.filter(ImageFilter.GaussianBlur(radius=0.35))
        target.parent.mkdir(parents=True, exist_ok=True)
        result.save(target, format="JPEG", quality=92)
    return dict(
        angle=angle, brightness=brightness, contrast=contrast,
        saturation=saturation, mirrored=mirrored, blur=blur, seed=seed,
    )


def build_dataset(
    old_zip: Path,
    good_zip: Path,
    destination: Path,
    *,
    expected_old: int = 12536,
    expected_good: int = 534,
    seed: int = 42,
    old_sha256: str = OLD_SHA256,
    good_sha256: str = GOOD_SHA256,
) -> dict:
    """Create new files only; preserve old split and group new good images by ID."""
    old_zip, good_zip, destination = map(Path, (old_zip, good_zip, destination))
    if destination.exists():
        raise FileExistsError(destination)
    if not zipfile.is_zipfile(old_zip) or not zipfile.is_zipfile(good_zip):
        raise ValueError("Both source files must be ZIP archives")
    hashes = {"old": file_sha256(old_zip), "good": file_sha256(good_zip)}
    if hashes != {"old": old_sha256, "good": good_sha256}:
        raise ValueError("Source ZIP SHA-256 differs from the audited input")

    with zipfile.ZipFile(old_zip) as old_archive, zipfile.ZipFile(good_zip) as good_archive:
        old_items = [(info, _old_entry(info.filename)) for info in _safe_members(old_archive)]
        old_items = [(info, entry) for info, entry in old_items if entry is not None]
        good_items = list(_safe_members(good_archive))
        if len(old_items) != expected_old or len(good_items) != expected_good:
            raise ValueError(f"Image counts changed: old={len(old_items)}, good={len(good_items)}")
        groups = defaultdict(list)
        for info in good_items:
            subject = PurePosixPath(info.filename).name.split("_", 1)[0]
            if not subject.isdecimal():
                raise ValueError(f"Missing good subject ID: {info.filename}")
            groups[subject].append(info.filename)
        allocation = split_good_subjects(groups, seed)
        destination.mkdir(parents=True)
        rows = []
        pixel_splits = {}
        for archive, items, source_name in (
            (old_archive, old_items, "old"),
            (good_archive, [(i, (allocation[i.filename.split('/')[-1].split('_', 1)[0]], "양호"))
                            for i in good_items], "good"),
        ):
            for info, (split, label) in items:
                basename = PurePosixPath(info.filename).name
                target = destination / "original" / split / label / basename
                if target.exists():
                    raise ValueError(f"Colliding image filename: {target}")
                data = archive.read(info)
                pixel_hash = _pixel_hash(data)
                previous = pixel_splits.get(pixel_hash)
                if previous is not None:
                    raise ValueError(f"Identical image appears twice: {previous} / {info.filename}")
                pixel_splits[pixel_hash] = info.filename
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
                rows.append(dict(
                    variant="original", split=split, class_name=label,
                    path=target.relative_to(destination).as_posix(), source_zip=source_name,
                    source_member=info.filename,
                    subject_id=basename.split("_", 1)[0] if source_name == "good" else "",
                    source_sha256=hashlib.sha256(data).hexdigest(),
                    augmentation="",
                ))

    train_by_class = defaultdict(list)
    for row in rows:
        if row["split"] == "train":
            train_by_class[row["class_name"]].append(row)
    for label in CLASSES:
        originals = sorted(train_by_class[label], key=lambda row: row["path"])
        if not originals:
            raise ValueError(f"Empty training class: {label}")
        if label == "양호":
            choices = [(row, index) for row in originals for index in range(1, 6)]
        else:
            rng = random.Random(seed + CLASSES.index(label))
            choices = [(row, 1) for row in rng.sample(originals, k=len(originals) // 2)]
        for row, index in choices:
            source = destination / row["path"]
            stem = PurePosixPath(row["path"]).stem
            name = f"{stem}_aug{index}.jpg"
            target = destination / "augmentation" / "train" / label / name
            if target.exists():
                raise ValueError(f"Colliding augmentation filename: {target}")
            transform_seed = int.from_bytes(
                hashlib.sha256(f"{seed}:{row['path']}:{index}".encode()).digest()[:8],
                "big",
            )
            params = _augment(source, target, transform_seed)
            rows.append(dict(
                variant="augmentation", split="train", class_name=label,
                path=target.relative_to(destination).as_posix(),
                source_zip=row["source_zip"], source_member=row["source_member"],
                subject_id=row["subject_id"], source_sha256=row["source_sha256"],
                augmentation=json.dumps(params, ensure_ascii=False, sort_keys=True),
            ))

    counts = Counter((row["variant"], row["split"], row["class_name"]) for row in rows)
    contract = dict(
        protocol="hair_six_class_v2", seed=seed, class_names=CLASSES,
        source_sha256=hashes, expected_old=expected_old, expected_good=expected_good,
        good_subjects=len(groups), good_group_assignment=allocation,
        original_counts={split: {label: counts[("original", split, label)] for label in CLASSES}
                         for split in SPLITS},
        augmented_extra_counts={label: counts[("augmentation", "train", label)]
                                for label in CLASSES},
        good_label_basis="user_confirmed_all_six_source_values_zero; source_json_unavailable",
        limitations=[
            "Old five-class filenames lost person/session IDs; person leakage remains unverified.",
            "Cross-ZIP byte and RGB-pixel audit found zero exact matches; "
            "near duplicates are not excluded.",
            "Augmented training changes both image variation and class counts.",
        ],
    )
    with (destination / "image_manifest.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    (destination / "dataset_contract.json").write_text(
        json.dumps(contract, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return contract


def archive_dataset(dataset: Path, target: Path) -> str:
    if target.exists():
        raise FileExistsError(target)
    with zipfile.ZipFile(target, "x", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for path in sorted(dataset.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(dataset.parent).as_posix())
    return file_sha256(target)


def dataset_factory(root: Path, variant: str, batch_size: int, seed: int = 42):
    """Make fixed-order evaluation and weighted training datasets from one v2 root."""
    import numpy as np
    import tensorflow as tf

    class_to_index = {name: index for index, name in enumerate(CLASSES)}

    def make(split: str, size: int, is_train: bool):
        paths, labels = [], []
        for label in CLASSES:
            folders = [root / "original" / split / label]
            if is_train and variant == "augmented":
                folders.append(root / "augmentation" / "train" / label)
            for folder in folders:
                files = [p for p in sorted(folder.iterdir()) if p.is_file()]
                paths.extend(str(p) for p in files)
                labels.extend([class_to_index[label]] * len(files))
        if not paths or (is_train and set(labels) != set(range(len(CLASSES)))):
            raise ValueError("Missing dataset images or class")
        labels_array = np.asarray(labels, dtype=np.int32)
        dataset = tf.data.Dataset.from_tensor_slices((paths, labels_array))
        if is_train:
            dataset = dataset.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
            counts = np.bincount(labels_array, minlength=len(CLASSES))
            weights = np.sqrt(counts.mean() / counts)
            weights /= np.mean(weights[labels_array])

        def decode(path, label):
            image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
            image.set_shape((None, None, 3))
            image = tf.image.resize(tf.cast(image, tf.float32), (size, size))
            target = tf.one_hot(label, len(CLASSES))
            if is_train:
                return image, target, tf.gather(tf.constant(weights, tf.float32), label)
            return image, target

        dataset = dataset.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
        dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        return dataset, paths

    return make


def run_four(
    dataset: Path,
    output: Path,
    *,
    data_sha256: str,
    batch_size: int = 32,
    stage1_epochs: int = 15,
    stage2_epochs: int = 15,
    seed: int = 42,
    code_hashes: dict | None = None,
    code_commit: str = "",
) -> dict:
    """Run/resume four trials; select by validation macro F1, then test once."""
    import platform

    import keras
    import matplotlib.pyplot as plt
    import numpy as np
    import tensorflow as tf

    from mediflow_datasets import common_engine as engine

    contract = json.loads((dataset / "dataset_contract.json").read_text(encoding="utf-8"))
    if contract["class_names"] != CLASSES:
        raise ValueError("Six-class order differs from the fixed contract")
    settings = dict(
        protocol="hair_six_class_four_v1", dataset_sha256=data_sha256,
        dataset_contract=hashlib.sha256(
            (dataset / "dataset_contract.json").read_bytes()
        ).hexdigest(),
        trials=TRIALS, seed=seed, batch_size=batch_size,
        stage1_epochs=stage1_epochs, stage2_epochs=stage2_epochs,
        keras=keras.__version__, tensorflow=tf.__version__, numpy=np.__version__,
        python=platform.python_version(), code_hashes=code_hashes or {},
    )
    signature = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()
    output.mkdir(parents=True, exist_ok=True)
    config_path = output / "run_config.json"
    if config_path.exists():
        if json.loads(config_path.read_text(encoding="utf-8"))["signature"] != signature:
            raise ValueError("RESUME_DIR has different data, code settings, or environment")
    else:
        engine.write_json(config_path, {
            "signature": signature, "settings": settings,
            "code_commit_at_generation": code_commit,
            "code_state": "exact embedded source copies saved; may include uncommitted changes",
        })
        engine.write_json(output / "class_names.json", CLASSES)
        shutil.copyfile(dataset / "dataset_contract.json", output / "dataset_contract.json")
    records = []
    for trial in TRIALS:
        spec = {**trial, "class_count": len(CLASSES)}
        factory = dataset_factory(dataset, trial["variant"], batch_size, seed)
        print("실험 시작:", trial["id"])
        records.append(engine.run_trial(
            spec, factory, output, signature, seed=seed,
            epochs1=stage1_epochs, epochs2=stage2_epochs,
        ))
        print("Validation Macro F1:", records[-1]["validation"]["macro_f1"])

    winner = max(records, key=lambda row: (
        row["validation"]["macro_f1"], row["validation"]["accuracy"]
    ))
    summary = dict(
        protocol="hair_six_class_four_v1", selection="validation_macro_f1_then_accuracy",
        winner=winner["id"], test_evaluated=False,
        experiments=[{
            "id": row["id"], "validation_accuracy": row["validation"]["accuracy"],
            "validation_macro_f1": row["validation"]["macro_f1"],
            "validation_good_recall": row["validation"]["recall"][CLASSES.index("양호")],
            "training_seconds": row["training_seconds"],
            "model_bytes": row["model_bytes"],
            "selected_stage": row["selected_stage"],
        } for row in records],
    )
    engine.write_json(output / "comparison_summary.json", summary)
    with (output / "validation_comparison.csv").open(
        "w", encoding="utf-8-sig", newline=""
    ) as stream:
        writer = csv.DictWriter(stream, fieldnames=list(summary["experiments"][0]))
        writer.writeheader()
        writer.writerows(summary["experiments"])
    fig, axes = plt.subplots(4, 2, figsize=(16, 18))
    for row_index, record in enumerate(records):
        history = record["history"]
        for col, key in enumerate(("accuracy", "loss")):
            axes[row_index, col].plot(history[key], label="Train")
            axes[row_index, col].plot(history["val_" + key], label="Validation")
            axes[row_index, col].axvline(record["stage_boundary"] - 0.5, color="gray", ls="--")
            axes[row_index, col].set_title(f"{record['id']} — {key}")
            axes[row_index, col].set_xlabel("Epoch")
            axes[row_index, col].grid(alpha=0.2)
            axes[row_index, col].legend()
    fig.tight_layout()
    fig.savefig(output / "four_training_curves.png", dpi=170)
    plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    names = [r["id"] for r in summary["experiments"]]
    axes[0].bar(names, [r["validation_accuracy"] for r in summary["experiments"]])
    axes[1].bar(names, [r["validation_macro_f1"] for r in summary["experiments"]])
    for ax, title in zip(axes, ("Validation Accuracy", "Validation Macro F1"), strict=True):
        ax.set(title=title, ylim=(0, 1))
        ax.tick_params(axis="x", labelrotation=30)
    fig.tight_layout()
    fig.savefig(output / "validation_dashboard.png", dpi=170)
    plt.close(fig)

    test_path = output / "selected_test_metrics.json"
    if not test_path.exists():
        keras.backend.clear_session()
        model = keras.models.load_model(engine.selected_model_path(output, winner), compile=False)
        factory = dataset_factory(dataset, winner["spec"]["variant"], batch_size, seed)
        test_data, test_paths = factory("test", winner["spec"]["size"], False)
        metrics = engine.evaluate_to_files(model, test_data, test_paths, output, "selected_test")
        metrics["selected_model"] = winner["id"]
        engine.write_json(test_path, metrics)
        del model
    else:
        metrics = engine.read_json(test_path)
    summary["test_evaluated"] = True
    summary["selected_test_accuracy"] = metrics["accuracy"]
    summary["selected_test_macro_f1"] = metrics["macro_f1"]
    selected_path = engine.selected_model_path(output, winner)
    summary["selected_model_sha256"] = engine.file_hash(selected_path)
    engine.write_json(output / "comparison_summary.json", summary)
    with (output / "validation_class_f1.csv").open(
        "w", encoding="utf-8-sig", newline=""
    ) as stream:
        writer = csv.writer(stream)
        writer.writerow(["experiment", *CLASSES])
        for row in records:
            writer.writerow([row["id"], *row["validation"]["class_f1"]])
    fig, ax = plt.subplots(figsize=(10, 5))
    matrix = np.array([row["validation"]["class_f1"] for row in records])
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
    ax.set(xticks=range(len(CLASSES)), xticklabels=CLASSES,
           yticks=range(len(records)), yticklabels=[row["id"] for row in records],
           title="Validation class F1")
    for index in range(len(records)):
        for column in range(len(CLASSES)):
            ax.text(column, index, f"{matrix[index, column]:.2f}",
                    ha="center", va="center")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(output / "validation_class_f1.png", dpi=170)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 6))
    confusion = np.asarray(metrics["confusion_matrix"])
    image = ax.imshow(confusion, cmap="Blues")
    ax.set(xticks=range(len(CLASSES)), xticklabels=CLASSES,
           yticks=range(len(CLASSES)), yticklabels=CLASSES,
           xlabel="Predicted", ylabel="True", title="Selected model — Test")
    for row in range(len(CLASSES)):
        for column in range(len(CLASSES)):
            ax.text(column, row, str(confusion[row, column]), ha="center", va="center")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(output / "selected_test_confusion.png", dpi=170)
    plt.close(fig)
    report_zip = output.parent / (output.name + "_reports.zip")
    if not report_zip.exists():
        with zipfile.ZipFile(report_zip, "x", zipfile.ZIP_DEFLATED) as archive:
            for path in output.rglob("*"):
                if path.is_file() and path.suffix != ".keras":
                    archive.write(path, path.relative_to(output.parent).as_posix())
    return {
        "summary": summary, "report_zip": str(report_zip),
        "output": str(output), "selected_model": str(selected_path),
    }


In [ ]:
import sys, types, hashlib
SOURCES = {
    'common_engine': (_EMBEDDED_SOURCE_DIR / 'common_engine.py').read_text(encoding='utf-8'),
    'hair_six_class_suite': (_EMBEDDED_SOURCE_DIR / 'hair_six_class_suite.py').read_text(encoding='utf-8'),
}
import hashlib as _source_hashlib
_EXPECTED_SOURCE_SHA256 = {'common_engine': '9bb8b0bbbc1e293d45792a8263903f322051aac1209c536c8dbc11a40c7b1038', 'hair_six_class_suite': '1ba2978cb82ab91ab8170b6675048e651d67683f8046faf0db8f8dfc7edb7b28'}
for _name, _code in SOURCES.items():
    if _source_hashlib.sha256(_code.encode('utf-8')).hexdigest() != _EXPECTED_SOURCE_SHA256[_name]:
        raise ValueError(f'내장 코드가 원본과 다릅니다: {_name}. 셀을 다시 실행하세요.')
BUILD_COMMIT = 'eb98690ba46983736012150ed0e705dcaa6fe1f7'
SOURCE_HASHES = {
    name: hashlib.sha256(code.encode()).hexdigest() for name, code in SOURCES.items()
}
package = types.ModuleType('mediflow_datasets')
package.__path__ = []
sys.modules['mediflow_datasets'] = package
for name, code in SOURCES.items():
    module = types.ModuleType('mediflow_datasets.' + name)
    sys.modules[module.__name__] = module
    exec(compile(code, name + '.py', 'exec'), module.__dict__)
from mediflow_datasets import hair_six_class_suite as suite
print('내장 코드 해시:', SOURCE_HASHES)


## 4. 6클래스 데이터 구성

기존 ZIP은 유지하고 새 ZIP을 생성합니다.


In [ ]:
from mediflow_datasets import hair_six_class_suite as suite

source_hashes = {
    'old': suite.OLD_SHA256,
    'good': suite.GOOD_SHA256,
    'builder_code': SOURCE_HASHES['hair_six_class_suite'],
    'seed': SEED,
}
name_digest = hashlib.sha256(json.dumps(source_hashes, sort_keys=True).encode()).hexdigest()[:12]
DATASET_ZIP = dataset_dir / ('hair_clean_v2_6class_' + name_digest + '.zip')
SHA_FILE = DATASET_ZIP.with_name(DATASET_ZIP.name + '.sha256')
free = shutil.disk_usage('/content').free
if free < 16 * 1024**3:
    raise RuntimeError('Colab 작업 공간 16GiB 이상이 필요합니다. 새 런타임을 사용하세요.')
work = Path('/content') / ('hair_six_class_' + uuid.uuid4().hex[:8])
work.mkdir()
local_dataset = work / 'dataset'
if DATASET_ZIP.exists():
    if not SHA_FILE.is_file():
        raise ValueError(
            '기존 v2 ZIP의 SHA-256 확인 파일이 없습니다. 파일을 수정하지 말고 점검하세요.'
        )
    expected = SHA_FILE.read_text().strip().split()[0]
    local_zip = work / DATASET_ZIP.name
    shutil.copyfile(DATASET_ZIP, local_zip)
    if suite.file_sha256(local_zip) != expected:
        raise ValueError('기존 v2 ZIP SHA-256이 다릅니다.')
    with zipfile.ZipFile(local_zip) as archive:
        for info in archive.infolist():
            path = Path(info.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in info.filename:
                raise ValueError('안전하지 않은 v2 ZIP 항목: ' + info.filename)
        archive.extractall(work)
    data_hash = expected
else:
    local_old = work / 'hair_datasets.zip'
    local_good = work / 'hair_good_raw_v1.zip'
    shutil.copyfile(OLD_ZIP, local_old)
    shutil.copyfile(GOOD_ZIP, local_good)
    contract = suite.build_dataset(local_old, local_good, local_dataset, seed=SEED)
    local_zip = work / DATASET_ZIP.name
    data_hash = suite.archive_dataset(local_dataset, local_zip)
    shutil.copyfile(local_zip, DATASET_ZIP)
    if suite.file_sha256(DATASET_ZIP) != data_hash:
        raise IOError('Drive에 저장한 v2 ZIP 해시가 다릅니다.')
    SHA_FILE.write_text(data_hash + '  ' + DATASET_ZIP.name + '\n')
print('새 6클래스 데이터 ZIP:', DATASET_ZIP)
print('새 데이터 SHA-256:', data_hash)
print(json.dumps(json.loads((local_dataset / 'dataset_contract.json').read_text()),
                 ensure_ascii=False, indent=2))


## 5. 4개 실험·그래프·선정 모델 Test

끊기면 RESUME_DIR을 설정하세요.


In [ ]:
from mediflow_datasets import hair_six_class_suite as suite

parent = PROJECT_ROOT / '2_results' / 'hair'
parent.mkdir(parents=True, exist_ok=True)
if RESUME_DIR:
    OUTPUT_DIR = Path(RESUME_DIR)
    if not OUTPUT_DIR.is_dir() or OUTPUT_DIR.parent != parent:
        raise ValueError('RESUME_DIR은 이 프로젝트의 Hair 결과 폴더여야 합니다.')
else:
    run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
    OUTPUT_DIR = parent / ('hair_six_class_four_' + run_id)
    OUTPUT_DIR.mkdir(exist_ok=False)
for name, code in SOURCES.items():
    target = OUTPUT_DIR / (name + '.py')
    if target.exists() and target.read_text(encoding='utf-8') != code:
        raise ValueError('저장된 코드와 현재 노트북 코드가 다릅니다: ' + name)
    if not target.exists():
        target.write_text(code, encoding='utf-8')
print('결과 폴더 / 중단 시 RESUME_DIR:', OUTPUT_DIR)
result = suite.run_four(
    local_dataset, OUTPUT_DIR, data_sha256=data_hash,
    batch_size=BATCH_SIZE, stage1_epochs=STAGE1_EPOCHS,
    stage2_epochs=STAGE2_EPOCHS, seed=SEED,
    code_hashes=SOURCE_HASHES, code_commit=BUILD_COMMIT,
)
print(json.dumps(result['summary'], ensure_ascii=False, indent=2))
print('모델 없는 보고서 ZIP:', result['report_zip'])
print('선정 모델 파일:', result['selected_model'])
